# Chapter 19 – Worked examples

This notebook runs the examples of [Chapter 19](https://katra.ballardini.com.ar/power-grid-topology-optimization-course/devdev/part-3-power-systems/ch19-operational-security/) live. The code lives in the modules next to it
(`metrics.py`, `case7.py`); the cells only call them, so a change in the module is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
# JupyterLite only: install the libraries this notebook and its chapter modules use.
%pip install -q colorama==0.4.6 deepdiff==8.6.2 geojson==3.3.0 networkx numpy packaging pandas pandera==0.34.1 scipy tqdm typing_extensions tzdata
%pip install -q --no-deps pandapower==3.5.6

In [ ]:
import case7
import metrics
import numpy as np

## ToOp's DC security metrics on three branches

Three monitored branches, one timestep, two contingencies. `overload_energy_n1` takes, per branch, the worst
contingency, then sums over branches: $20 + 10 + 5 = 35$ MW. The power law weights the same overloads as
$(\text{rel}^{1.5} - 1)\, f_\max$.

In [ ]:
print('overload energy N-1 :', metrics.overload_energy_n1(metrics.n_1, metrics.f_max))
print('power law N-1       :', round(metrics.power_law_n1(metrics.n_1, metrics.f_max), 2))
print('critical branches   :', metrics.critical_branch_count_n1(metrics.n_1, metrics.f_max))
print('double limits       :', metrics.double_limits(metrics.n_0[:, None, :], metrics.f_max))
print('N-0 to N-1 delta    :', metrics.n0_n1_delta(metrics.n_0, metrics.n_1))

Change one entry of the N-1 matrix and watch which metric moves: a larger overload on an already critical
branch changes the energy but not the count.

In [ ]:
n_1 = metrics.n_1.copy()
n_1[0, 1, 0] = 140.0  # branch 0, second contingency: 120 -> 140 MW on a 100 MW branch
print(metrics.overload_energy_n1(n_1, metrics.f_max), metrics.critical_branch_count_n1(n_1, metrics.f_max))

## The price of N-1 security on the 7-bus case

`dispatch()` solves the OPF, `dispatch(range(N_LINE))` the SCOPF with every single-line outage as a
constraint. The &#36;100/h difference is the price of security.

In [ ]:
opf, scopf = case7.dispatch(), case7.dispatch(range(case7.N_LINE))
for name, r in (('OPF', opf), ('SCOPF', scopf)):
    print(f'{name:5s} {r.fun:6.0f} $/h  Pg {np.round(r.x, 1)}  worst N-1 {case7.worst(case7.n1_loading(r.x))}')

The whole post-contingency loading table of the OPF dispatch, in percent (rows: outaged line, columns: line).
Only the two lines leaving bus 1 exceed 100 %.

In [ ]:
pct = case7.n1_loading(opf.x)
print('        ' + ' '.join(f'{n:>6s}' for n in case7.NAMES))
for name, row in zip(case7.NAMES, pct, strict=True):
    print(f'{name:>6s}  ' + ' '.join(f'{v:6.1f}' for v in row))

## DC security is not AC security

The same two dispatches through pandapower's AC power flow, one outage at a time. Loading now includes the
reactive flow and the losses, which the DC model drops.

In [ ]:
for name, r in (('OPF', opf), ('SCOPF', scopf)):
    ac = case7.pandapower_n1(case7.pandapower_net(r.x), ac=True)
    print(f'{name:5s} worst AC N-1 loading {ac.max():.1f} % on {case7.NAMES[int(ac.argmax())]}')

Try it yourself: raise the limit of lines 1–2 and 1–3 to 260 MW (`case7.LIMIT.copy()`, then pass `limit=`
to `dispatch` and `n1_loading`) and check that the OPF dispatch becomes N-1 secure for free.

In [ ]:
lim = case7.LIMIT.copy()
lim[[0, 1]] = 260
print(case7.worst(case7.n1_loading(case7.dispatch((), lim).x, lim)))